In [13]:
import os
print(os.getcwd())

/mnt/shared-storage-gpfs2/beam-gpfs02/yulang/master/nemo_cellflow/preprocessing/arcinstitute/datasets/State_Replogle_Filtered


In [32]:
# 读取TOML文件
from pathlib import Path
import tomllib
few_shot_name = "rpe1"
toml_path = Path(f"few_shot/{few_shot_name}/{few_shot_name}.toml")  # 改成你的路径

with toml_path.open("rb") as f:
    cfg = tomllib.load(f)
    
cfg["fewshot"].keys()

dict_keys(['replogle.rpe1'])

In [33]:
# celltype and drug in val and test set
from itertools import product
val_cartesian_fewshot_list = []
test_cartesian_fewshot_list = []
for x in cfg["fewshot"].keys():
    cell_name = x.split(".")[-1]
    val_drug_names = cfg["fewshot"][x]["val"]
    val_cartesian_fewshot = list(product([cell_name], val_drug_names))
    test_drug_names = cfg["fewshot"][x]["test"]
    test_cartesian_fewshot = list(product([cell_name], test_drug_names))
    val_cartesian_fewshot_list.extend(val_cartesian_fewshot)
    test_cartesian_fewshot_list.extend(test_cartesian_fewshot)

print(val_cartesian_fewshot_list[:10])
print(test_cartesian_fewshot_list[-10:])
print(len(val_cartesian_fewshot_list))
print(len(test_cartesian_fewshot_list))

[('rpe1', 'RPA3'), ('rpe1', 'POLR1E'), ('rpe1', 'TMEM258'), ('rpe1', 'PPIH'), ('rpe1', 'SMC2'), ('rpe1', 'NUBP2'), ('rpe1', 'RNF8'), ('rpe1', 'ATF4'), ('rpe1', 'USPL1'), ('rpe1', 'SDHA')]
[('rpe1', 'EEFSEC'), ('rpe1', 'AHCTF1'), ('rpe1', 'CLNS1A'), ('rpe1', 'TAF8'), ('rpe1', 'DOHH'), ('rpe1', 'NUP35'), ('rpe1', 'RPUSD4'), ('rpe1', 'EP400'), ('rpe1', 'IFITM2'), ('rpe1', 'THOC7')]
1061
1061


In [34]:
import pickle
with open("pert_sample_list.pkl", "rb") as f:
    pert_data = pickle.load(f)
    
from tqdm import tqdm
train_sample_list = []
val_sample_list = []
test_sample_list = []
for x in tqdm(pert_data):
    cell_drug_product = (x["cartesian_key"][1], x["cartesian_key"][2])
    if cell_drug_product in val_cartesian_fewshot_list:
        val_sample_list.append(x)
        test_sample_list.append(x)
    elif cell_drug_product in test_cartesian_fewshot_list:
        test_sample_list.append(x)
    else:
        train_sample_list.append(x)

len(val_sample_list)

100%|██████████| 7623/7623 [00:00<00:00, 49550.08it/s]


1455

In [35]:
# 分别写入lmdb
import lmdb
import pickle
from tqdm import tqdm

def dump_list_to_lmdb(
    data_list,
    lmdb_path: str,
    map_size_bytes: int,   # 必须足够大（比如原始数据大小 * 1.2~2）
    protocol: int = pickle.HIGHEST_PROTOCOL,
):
    env = lmdb.open(
        lmdb_path,
        map_size=map_size_bytes,
        subdir=True,
        create=True,
        lock=True,
        readahead=False,
        max_dbs=1,
    )

    with env.begin(write=True) as txn:
        # 存长度
        txn.put(b"__len__", str(len(data_list)).encode("utf-8"))

    # 分批写：避免一个 txn 太大
    batch = 10_000
    for start in tqdm(range(0, len(data_list), batch), desc="writing lmdb"):
        end = min(start + batch, len(data_list))
        with env.begin(write=True) as txn:
            for i in range(start, end):
                k = str(i).encode("utf-8")
                v = pickle.dumps(data_list[i], protocol=protocol)
                txn.put(k, v)

    env.sync()
    env.close()
def fewshot_lmdb_dir(split="train"):
    base_dir = f"few_shot/{few_shot_name}"
    lmdb_dir = os.path.join(base_dir, f"replogle_{split}_{few_shot_name}")
    return lmdb_dir

lmdb_dict = {fewshot_lmdb_dir("train"): train_sample_list, fewshot_lmdb_dir("val"): val_sample_list, fewshot_lmdb_dir("test"): test_sample_list}
for key, data_list in lmdb_dict.items():
    dump_list_to_lmdb(data_list, key, map_size_bytes=180 * (1 << 30))
    
    


writing lmdb:   0%|          | 0/1 [00:00<?, ?it/s]

writing lmdb: 100%|██████████| 1/1 [00:00<00:00,  1.08it/s]


In [36]:
import pickle
with open("control_dict.pkl", "rb") as f:
    control_data = pickle.load(f)
list(control_data.keys())

[('0', 'hepg2', 'non-targeting'),
 ('0', 'jurkat', 'non-targeting'),
 ('0', 'k562', 'non-targeting'),
 ('0', 'rpe1', 'non-targeting')]

In [37]:
import pickle
with open("control_dict.pkl", "rb") as f:
    control_data = pickle.load(f)
    
    
# 字符串，tuple互相转换脚本
import ast
test_tuple = list(control_data.keys())[0]
test_str = str(test_tuple)
print(len(test_tuple))
print(len(test_str))
print(len(ast.literal_eval(test_str)))

def dump_dict_to_lmdb(
    data_dict,
    lmdb_path: str,
    map_size_bytes: int,
    protocol: int = pickle.HIGHEST_PROTOCOL,
):

    env = lmdb.open(
        lmdb_path,
        map_size=map_size_bytes,
        subdir=True,
        create=True,
        lock=True,
        readahead=False,
        max_dbs=1,
    )

    keys = list(data_dict.keys())

    with env.begin(write=True) as txn:
        txn.put(b"__len__", str(len(keys)).encode("utf-8"))
        txn.put(b"__keys__", pickle.dumps(keys, protocol=protocol))

        for k_str, obj in tqdm(data_dict.items()):
            k = str(k_str).encode("utf-8")
            v = pickle.dumps(obj, protocol=protocol)
            txn.put(k, v)

    env.sync()
    env.close()

# 写入lmdb
dump_dict_to_lmdb(control_data, fewshot_lmdb_dir("control"), map_size_bytes=180 * (1 << 30))

3
31
3


100%|██████████| 4/4 [00:00<00:00, 17.71it/s]


In [8]:
# read_lmdb_test
import pickle
import lmdb
pert_lmdb_path = "/mnt/shared-storage-gpfs2/beam-gpfs02/yulang/master/nemo_cellflow/preprocessing/arcinstitute/datasets/State_Replogle_Filtered/replogle_val"

control_pert = "non-targeting"
pert_env = lmdb.open(pert_lmdb_path, readonly=True)


processed_cartesian_keys = []
control_cartesian_keys = []
with pert_env.begin(write=False) as pert_txn:
    train_len = int(pert_txn.get(b"__len__"))
    print(train_len)
    for i in range(train_len):
        pert_buf = pert_txn.get(str(i).encode("utf-8"))
        pert_group = pickle.loads(pert_buf)
        pert_cartesian_key = pert_group["cartesian_key"]
        control_cartesian_key = (pert_cartesian_key[0], pert_cartesian_key[1], control_pert)
        processed_cartesian_keys.append(pert_cartesian_key)
        control_cartesian_keys.append(control_cartesian_key)
pert_env.close()
pert_group

1048


{'cartesian_key': ('0', 'hepg2', 'PLEKHN1'),
 'cell_matrix': <Compressed Sparse Row sparse matrix of dtype 'float32'
 	with 123212 stored elements and shape (128, 2000)>}

In [3]:
with control_env.begin(write=False) as control_txn:
    control_buf = control_txn.get(str(control_cartesian_key).encode())
    control_group = pickle.loads(control_buf)
    print(int(control_txn.get(b"__len__")))
control_group


4


<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 10494012 stored elements and shape (11485, 2000)>

In [3]:
len(control_cartesian_keys)

4814